# NCDB source and quality audit

## tl;dr
This companion reruns or verifies the full-file audit for the **23 selected annual releases, 1999–2021**, and separately displays the older 2019 CSV comparator. The assessment and exact-value tables live in `docs/data_quality_assessment.md` and `reports/tables/`. No model is trained and no rows are silently deduplicated.

## Context & Methods
The unit of a raw record is a person/vehicle segment, including dummy and unresolved segments. Collision counts group the candidate `(C_YEAR, C_CASE)` within a single release. Identified-person checks additionally require numeric `V_ID` and `P_ID` in 1–99. `V_ID=99` denotes the pedestrian dummy vehicle; it is not a real motor vehicle.

### Key assumptions
`C_CASE` is a candidate event identifier whose formal semantics are absent from the supplied dictionary. Collision attributes must agree across linked segments. Numeric code padding is normalized only in a temporary audit view. `P_USER=U` combines not-stated, other and unknown, so these meanings cannot be separated. `*`/`**` have no verified meaning in the linked dictionary. No observation is inferred to be a non-accident example.

The audit implementation is [audit_data.py](../src/road_to_safety/audit_data.py). Its checks, source fingerprints and pinned runtime versions are retained in `data/derived/quality_audit.json`. All source records are scanned; openpyxl checks of the first 100 XLSX records are an independent parser comparison, not a sampled data-quality assessment. Tables provide exact annual reconciliation and code-rate lookup; charts are deferred to exploratory analysis.

### Set up the project runtime
From the project directory run `uv sync --locked --extra notebook`, then execute this notebook using the project `.venv/bin/python` kernel. The working directory may be the project root or `notebooks/`. The files in `data/sources.json` must already have been acquired. The saved outputs were executed with a temporary kernel specification pointing to that project interpreter; no global kernel installation is required.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/sources.json").exists(), "Run from project root or notebooks/"
sys.path.insert(0, str(ROOT / "src"))
from road_to_safety.audit_data import run, sha256


## Data
### Verify originals and run every selected year
The cache is reused only if source SHA256 and audit-code SHA256 match. A mismatch with the acquisition manifest fails the audit. Two workers bound memory use. The older 2019 CSV stays outside all selected-release totals.

In [2]:
results = run(ROOT, workers=2, include_comparator=True)
manifest = json.loads((ROOT / "data/sources.json").read_text())
selected_sources = [r for r in manifest["resources"] if r.get("selected") and r.get("year") is not None]
selected = [r for r in results if r["selected"]]
assert len(selected) == 23
assert {r["expected_year"] for r in selected} == set(range(1999, 2022))
assert {r["path"] for r in selected} == {r["local_path"] for r in selected_sources}
assert all(sha256(ROOT / r["local_path"]) == r["sha256"] for r in selected_sources)
assert all(r["columns"] == 23 and r["unexpected_year_rows"] == 0 for r in selected)
print(f"Verified {len(selected)} selected releases and {len(results)-len(selected)} comparator.")
print(json.loads((ROOT / "data/derived/quality_audit.json").read_text())["runtime"])


Verified 23 selected releases and 1 comparator.
{'duckdb': '1.5.3', 'pandas': '2.3.3', 'openpyxl': '3.1.5', 'python-calamine': '0.9.0'}


## Results
### Annual collision targets
`C_SEV=1` means a collision producing at least one fatality; `2` means a collision producing nonfatal injury. These are collision counts, never a sum of repeated person-record severity labels. Person `P_ISEV=1/2/3` means no injury/injury/fatality, a separate outcome.

In [3]:
annual = pd.DataFrame(selected).sort_values("expected_year")
annual[["expected_year", "rows", "collisions", "fatal_collisions", "injury_collisions", "severity_conflict_collisions"]]


,expected_year,rows,collisions,fatal_collisions,injury_collisions,severity_conflict_collisions
0,1999,413509,151315,2632,148683,0
1,2000,422075,155838,2548,153290,0
2,2001,409389,151438,2415,149023,0
3,2002,420008,156415,2583,153832,0
4,2003,407036,152980,2487,150493,0
5,2004,389050,147588,2438,145150,0
6,2005,386470,148111,2551,145560,0
7,2006,378523,145103,2586,142517,0
8,2007,368507,141070,2455,138615,0
9,2008,338268,129764,2193,127571,0


### Identity and consistency exceptions
Extra exact rows are not automatically extra people. Conflicting repeated numeric person keys must be resolved or explicitly excluded from a person-level modeling task. A fatal collision without a recorded fatal person is flagged for review, not automatically relabeled.

In [4]:
checks = ["invalid_collision_key_rows", "unresolved_person_identity_rows", "exact_duplicate_extra_rows",
          "duplicate_identified_person_keys", "person_severity_conflict_keys", "vehicle_attribute_conflict_keys",
          "fatal_collisions_without_recorded_fatal_person", "injury_collisions_with_recorded_fatal_person"]
annual[["expected_year"] + checks]


,expected_year,invalid_collision_key_rows,unresolved_person_identity_rows,exact_duplicate_extra_rows,duplicate_identified_person_keys,person_severity_conflict_keys,vehicle_attribute_conflict_keys,fatal_collisions_without_recorded_fatal_person,injury_collisions_with_recorded_fatal_person
0,1999,0,823,0,0,0,0,1,0
1,2000,0,751,0,0,0,0,0,0
2,2001,0,727,0,1,1,0,0,0
3,2002,0,806,0,0,0,0,0,0
4,2003,0,734,0,0,0,0,0,0
5,2004,0,627,0,0,0,0,0,0
6,2005,0,734,0,0,0,0,0,0
7,2006,0,786,0,0,0,0,0,0
8,2007,0,799,0,3,2,5,0,0
9,2008,0,785,0,3,2,0,0,0


### Coded missingness and other states
The denominator is **raw source records**, including dummy people and unresolved identity. Counts describe data availability, not population prevalence or accident risk. `other` is a stated alternative, not an unknown value.

In [5]:
missingness = pd.read_csv(ROOT / "reports/tables/quality_missingness.csv")
missingness = missingness[missingness["selected"]]
summary = missingness.groupby(["field", "value_class"], as_index=False)["rows"].sum()
summary["share_of_all_source_rows"] = summary["rows"] / annual["rows"].sum()
summary = summary[summary["value_class"] != "stated"].sort_values("rows", ascending=False)
summary.head(25)


,field,value_class,rows,share_of_all_source_rows
45,P_SAFE,unknown,835111,0.108037
42,P_SAFE,not_applicable,747656,0.096723
12,C_RCFG,unknown,602285,0.077916
9,C_RALN,unknown,527699,0.068267
31,P_AGE,unknown,499246,0.064586
60,V_YEAR,unknown,410474,0.053102
0,C_CONF,other,390566,0.050527
35,P_ISEV,not_applicable,372548,0.048196
58,V_YEAR,not_applicable,357033,0.046189
48,P_SEX,unknown,342193,0.044269


### The 2019 release revision
The newer XLSX is selected. The older CSV remains an audit comparator and is excluded from pooled totals. Distinct event identifiers are release-specific; different totals do not establish missing records, a stable cross-release join, or which release is closer to an independently defined benchmark.

In [6]:
pd.DataFrame([r for r in results if r["expected_year"] == 2019])[["path", "selected", "rows", "collisions", "fatal_collisions", "injury_collisions"]]


,path,selected,rows,collisions,fatal_collisions,injury_collisions
0,data/raw/ncdb/2019_dataset_en.csv,False,272301,105782,1623,104159
1,data/raw/ncdb/y_2019_en.xlsx,True,269328,104630,1620,103010


### Capability checks
Schema inspection, rather than inferred IDs, governs geographic and calendar capabilities.

In [7]:
headers = {column for result in selected for column in result["original_headers"]}
print("Headers observed:", sorted(headers))
print("Selected totals:", annual[["rows", "collisions", "fatal_collisions", "injury_collisions"]].sum().to_dict())
print("Road-user definitions: 1 driver; 2 passenger; 3 pedestrian; 4 bicyclist; 5 motorcyclist; U not stated/other/unknown.")


Headers observed: ['C_ANNÉE', 'C_CAS', 'C_CASE', 'C_CFG', 'C_CFGR', 'C_CIRC', 'C_CONF', 'C_GRAVITÉ', 'C_HEURE', 'C_HOUR', 'C_JSEM', 'C_MNTH', 'C_MOIS', 'C_MÉTEO', 'C_RALN', 'C_RCFG', 'C_RSUR', 'C_SEV', 'C_TRAF', 'C_TRAR', 'C_VEHS', 'C_VÉH', 'C_WDAY', 'C_WTHR', 'C_YEAR', 'C_ÉCHSÉ', 'P_AGE', 'P_ID', 'P_ISEV', 'P_LOCP', 'P_PSN', 'P_SAFE', 'P_SEX', 'P_SEXE', 'P_SÉCURITÉ', 'P_USAG', 'P_USER', 'P_ÂGE', 'V_ANNÉE', 'V_ID', 'V_TYPE', 'V_YEAR']
Selected totals: {'rows': 7729886, 'collisions': 2949846, 'fatal_collisions': 47793, 'injury_collisions': 2902053}
Road-user definitions: 1 driver; 2 passenger; 3 pedestrian; 4 bicyclist; 5 motorcyclist; U not stated/other/unknown.


## Takeaways
The selected releases support a collision-severity feasibility study conditional on a recorded fatal/injury collision, subject to the documented identity, feature availability, revision and coverage limitations. The raw files support national annual/month/weekday/hour summaries. They do not contain province, coordinates or a complete calendar date, so they cannot support event hotspot maps, road-segment weather joins or accident-occurrence probabilities. Province-level aggregate maps require separate compatible published series; occurrence modeling requires exposure and a valid zero-crash observation frame.

See [the full assessment](../docs/data_quality_assessment.md) for quantified findings and required decisions before modeling. Originals and their acquisition provenance are recorded in [the source manifest](../data/sources.json); full audit evidence is retained in [the machine summary](../data/derived/quality_audit.json).